# 🎬 Gurukuladesam Batch Video Render Pipeline (NVIDIA L4 GPU NVENC)

High-speed parallel batch rendering powered by **NVIDIA Ada Lovelace L4 (24GB VRAM)** with:
- Dedicated **8th Gen NVENC (H.264 / AV1)** hardware video encoder.
- Dynamic 5–7s transitions with 1.0s `xfade` (cross-dissolve).
- 4-way cinematic pans + 3D parabolic curved arc camera moves.
- 100% upward divine drizzle / vibhuti flakes atmospheric composite.


In [ ]:
# Step 1: Verify NVIDIA L4 GPU & Hardware NVENC Support
!nvidia-smi
!ffmpeg -f lavfi -i testsrc=duration=2:size=1280x720:rate=24 -c:v h264_nvenc -preset p4 -y /tmp/test_nvenc.mp4 2>&1 | tail -n 6
!ls -lh /tmp/test_nvenc.mp4


In [ ]:
# Step 2: Clone repository & install dependencies
import os
if not os.path.exists('/content/Gurukuladesam'):
    !git clone -b main https://github.com/shishyan/GuruKulaDesam.git /content/Gurukuladesam
else:
    !cd /content/Gurukuladesam && git pull origin main

%cd /content/Gurukuladesam
!pip install -q Pillow numpy tqdm requests mutagen


In [ ]:
# Step 3: Fast Audio & Image Sync for Chapter 4
import requests, json, base64
from pathlib import Path

tunnel_url = 'https://a89df53bc954449f-49-206-115-211.serveousercontent.com'

# Download and assemble Chapter 4 audio (14 parts)
audio_dest = Path('/content/audio/audio_ch004.wav')
audio_dest.parent.mkdir(parents=True, exist_ok=True)
if not audio_dest.exists() or audio_dest.stat().st_size < 50 * 1024 * 1024:
    print('Downloading audio_ch004 in 14 parts...')
    with open(audio_dest, 'wb') as f:
        for p in range(14):
            part_file = f'audio_ch004_part_{p:02d}.bin'
            r = requests.get(f'{tunnel_url}/{part_file}')
            f.write(r.content)
    print(f'✔ Assembled audio_ch004.wav ({audio_dest.stat().st_size / (1024*1024):.1f} MB)')

# Download and assemble Chapter 4 images (5 parts)
ch4_img_dir = Path('/content/Gurukuladesam/production/scene_images/ch004')
if not ch4_img_dir.exists() or len(list(ch4_img_dir.glob('*.jpg'))) < 10:
    print('Downloading ch004 image bundle in 5 parts...')
    bundle_bytes = bytearray()
    for p in range(5):
        r = requests.get(f'{tunnel_url}/ch004_part_{p:02d}.bin')
        bundle_bytes.extend(r.content)
    bundle = json.loads(bundle_bytes.decode('utf-8'))
    for rel_path, b64 in bundle.items():
        dest = Path('/content/Gurukuladesam') / rel_path
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(base64.b64decode(b64))
    print(f'✔ Unpacked {len(bundle)} images for Chapter 4')

# Download atmospheric loops if needed
for asset in ['upward_divine_drizzle_loop.mp4', 'sacred_lamp_dhoopa_overlay_1280x720.png']:
    dest = Path('/content/Gurukuladesam/production') / asset
    if not dest.exists():
        r = requests.get(f'{tunnel_url}/{asset}')
        dest.write_bytes(r.content)
        print(f'✔ Downloaded {asset}')


In [ ]:
# Step 4: High-Speed NVENC Master Render for Chapter 4
%cd /content/Gurukuladesam
!python tools/build_cinematic_720p_film.py --chapter 4 --gpu nvenc
!ls -lh renders/releases/thirukkural_ch004_cinematic_720p_master.mp4
